# 같은 질문으로 검색 개선 효과 판정하기

재정렬 후 평균이 올라도 특정 고객군의 정답이 밀릴 수 있다. 질문별 검색 순위와 답의 근거를 함께 비교한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 물어볼 질문을 먼저 고정하는 이유

설정을 바꾼 다음 쉬운 질문만 더하면 개선한 부분을 구분할 수 없다. 질문 ID, 기대하는 조항, 답해도 되는 범위와 문서 판본을 실험 전에 저장한다.

Q1에 두 조항이 모두 필요한데 한 개만 1위라면 첫 적중은 빠르지만 답에 필요한 근거 절반은 없다. 반대로 답을 보류해야 할 질문에는 정답 조항을 억지로 만들어 넣지 않는다.

복수 근거에는 찾은 비율을 확인하고, 첫 근거가 언제 보였는지도 따로 계산한다. 그 뒤 답변이 실제 검색한 문장을 인용했는지 읽는다.

## 한 지표만 보면 놓치는 변경

K=2로 고정한다. Q1은 A와 B가 모두 있어야 한다. Q2는 C 하나면 된다.

In [1]:
gold = {"Q1":{"A","B"}, "Q2":{"C"}}
base = {"Q1":["X","A"], "Q2":["C","D"]}
reranked = {"Q1":["A","B"], "Q2":["D","E"]}
def at_two(ids, required):
    top = ids[:2]
    rank = next((i for i, doc in enumerate(top, 1) if doc in required), None)
    return len(set(top) & required)/len(required), 1/rank if rank else 0

print(at_two(base["Q1"],gold["Q1"]))
print(at_two(reranked["Q1"],gold["Q1"]))

(0.5, 0.5)
(1.0, 1.0)


Q1의 근거 회수율과 첫 적중 역순위가 모두 좋아졌다. 아직 Q2는 비교하지 않았으므로 전체 정책을 바꿀 수 없다.

### 새로 실패한 질문 찾아내기

이전 설정에서 근거를 하나라도 찾았는데 후보 설정에서는 하나도 찾지 못한 질문을 고른다.

In [2]:
lost = []

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>lost = [q for q in gold if at_two(base[q],gold[q])[0] &gt; 0 and at_two(reranked[q],gold[q])[0] == 0]
print(lost)</code></pre><pre>[&#x27;Q2&#x27;]
</pre><p>Q2는 1위에 있던 C가 상위 두 개에서 사라졌다. 평균 점수와 함께 새로 실패한 질문의 원문을 보아야 한다.</p></details>

## 검색 성공과 인용 오류를 나누기

정답 ID가 검색됐어도 모델은 후보의 다른 문장을 읽고 답할 수 있다. 값이 맞는지와 인용이 그 값을 뒷받침하는지 따로 기록한다.

In [4]:
rules = {"A":"연장 1회", "B":"연장 2회"}
expected = "연장 1회"
response = {"answer":"연장 1회", "cite":"B"}
print(response["answer"] == expected, rules[response["cite"]] == response["answer"])

True False


첫 판정은 참, 두 번째는 거짓이다. 두 값을 곱한 점수 하나만 남기면 답 생성기의 문제인지 인용 선택기의 문제인지 다시 찾기 어렵다.

### 잘못된 인용과 정답 미검색을 서로 다른 열에 남기기

A가 후보에 있는 실행과 없는 실행에서 같은 응답을 평가한다. 어떤 실패가 검색 단계부터 있었는지 구별한다.

In [5]:
first_run = ["A","B"]
second_run = ["X","B"]

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>def diagnose(ranked, response):
    return {&quot;정답 검색&quot;:&quot;A&quot; in ranked, &quot;인용 지지&quot;:rules.get(response[&quot;cite&quot;]) == response[&quot;answer&quot;]}
print(diagnose(first_run, response))
print(diagnose(second_run, response))</code></pre><pre>{&#x27;정답 검색&#x27;: True, &#x27;인용 지지&#x27;: False}
{&#x27;정답 검색&#x27;: False, &#x27;인용 지지&#x27;: False}
</pre><p>첫 실행은 A를 찾고도 B를 인용했다. 두 번째는 A를 찾는 단계부터 실패했다. 같은 문장을 고치더라도 바꿀 지점은 다르다.</p></details>

## 다른 입력에 적용하기

직접 만든 질문 세트를 개발용과 확인용으로 분리한다. 청크 크기, K, 재정렬 가운데 하나만 바꾼 뒤 이전에 맞힌 질문도 다시 실행한다.

개선한 문항과 악화된 문항을 한 건씩 골라 검색 후보, 최종 인용, 추가 호출량을 설명한다. 같은 문서를 단어만 바꿔 양쪽 세트에 넣지 않았는지도 확인한다.